# Fine-Tuning IndoBERT untuk Klasifikasi Aspek Multi-Label
Latih model pre-trained IndoBERT (indobenchmark/indobert-base-p1) pada klasifikasi aspek multi-label menggunakan Binary Cross-Entropy Loss.

In [ ]:
# Install dependencies
# !pip install transformers accelerate datasets scikit-learn -q

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, Features, Sequence, Value
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import json

## 1. Load Data

In [ ]:
dataset_path = '../data/cleaned_reviews_with_aspect_labels.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'cleaned_reviews_with_aspect_labels.csv'

df = pd.read_csv(dataset_path).dropna(subset=['text_clean'])
print(f"Total data: {len(df)} baris")

## 2. Format Input & Targets

In [ ]:
aspect_cols = ['aspek_rasa', 'aspek_harga', 'aspek_pelayanan', 'aspek_kecepatan', 'aspek_kebersihan', 'aspek_stok', 'aspek_aplikasi']
df['labels'] = df[aspect_cols].values.tolist()

print("Contoh input ulasan:")
print(df['text_clean'].iloc[0])
print("Contoh target labels:")
print(df['labels'].iloc[0])

## 3. Inisialisasi Tokenizer

In [ ]:
model_name = 'indobenchmark/indobert-base-p1'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text_clean'], padding='max_length', truncation=True, max_length=128)

## 4. Split Train / Test & Konfigurasi Dataset

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Definisikan skema data untuk target float (kebutuhan BCE loss)
class_features = Features({
    'text_clean': Value('string'),
    'labels': Sequence(Value('float32'))
})

train_dataset = Dataset.from_pandas(train_df[['text_clean', 'labels']].reset_index(drop=True), features=class_features)
test_dataset = Dataset.from_pandas(test_df[['text_clean', 'labels']].reset_index(drop=True), features=class_features)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

## 5. Konfigurasi Model Multi-Label

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(aspect_cols),
    problem_type="multi_label_classification"
)

## 6. Metrik Evaluasi

In [ ]:
def multi_label_metrics(predictions, labels, threshold=0.5):
    sigmoid = 1 / (1 + np.exp(-predictions))
    y_pred = np.zeros(sigmoid.shape)
    y_pred[sigmoid >= threshold] = 1
    y_true = labels
    
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro')
    f1_macro_average = f1_score(y_true=y_true, y_pred=y_pred, average='macro')
    roc_auc = roc_auc_score(y_true, sigmoid, average = 'micro')
    accuracy = accuracy_score(y_true, y_pred)
    
    metrics = {
        'f1_micro': f1_micro_average,
        'f1_macro': f1_macro_average,
        'roc_auc': roc_auc,
        'accuracy': accuracy
    }
    return metrics

def compute_metrics(p):
    predictions, labels = p
    return multi_label_metrics(predictions=predictions, labels=labels)

## 7. Setup Training Arguments & Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_dir='./logs',
    logging_steps=50,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

## 8. Eksekusi Training

In [ ]:
trainer.train()

## 9. Simpan Model

In [ ]:
model_save_path = '../weights/indobert_aspect_multilabel'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model IndoBERT berhasil disimpan di: {model_save_path}")